# ExplaineR in R — Explain a breast cytology classifier
Train a random forest, check its held-out performance, and use ExplaineR to see how recorded cell characteristics contribute to individual predictions. This example uses the Wisconsin breast cytology data distributed with **mlbench**.

[ExplaineR on CRAN](https://cran.r-project.org/package=explainer) · [Source code](https://github.com/PERSIMUNE/explainer) · [Full tutorial](https://persimune.github.io/explainer/articles/explainer_tutorial_binary_classification.html)

Kaggle hosts this example in a Python notebook. The analysis runs in R, and the R code below is editable.


## 1. Set up R
Use a CPU session with Internet enabled. The first run installs a separate R environment and ExplaineR 1.0.3 from CRAN.


In [ ]:
import io
import os
from pathlib import Path
import subprocess
import tarfile
import urllib.request

WORK = Path("/kaggle/working/explainer")
WORK.mkdir(parents=True, exist_ok=True)
TOOLS = Path("/kaggle/temp/explainer-r")
TOOLS.mkdir(parents=True, exist_ok=True)
ENV = TOOLS / "env"
MAMBA = TOOLS / "micromamba"
R = ENV / "bin/Rscript"
env = dict(os.environ, MAMBA_ROOT_PREFIX=str(TOOLS / "cache"))

def execute(args):
    result = subprocess.run([str(arg) for arg in args], env=env, cwd=WORK,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        print(result.stdout[-10000:])
        result.check_returncode()
    return result.stdout

if not MAMBA.exists():
    with urllib.request.urlopen("https://micro.mamba.pm/api/micromamba/linux-64/latest", timeout=90) as response:
        binary = response.read()
    with tarfile.open(fileobj=io.BytesIO(binary), mode="r:bz2") as archive:
        MAMBA.write_bytes(archive.extractfile("bin/micromamba").read())
    MAMBA.chmod(0o755)

if not (TOOLS / "ready").exists():
    print("Installing R and analysis dependencies…", flush=True)
    execute([MAMBA, "create", "-y", "-p", ENV, "-c", "conda-forge",
             "r-base=4.4", "pandoc", "r-cvms", "r-data.table", "r-dplyr", "r-egg",
             "r-ggplot2", "r-ggpubr", "r-magrittr", "r-plotly", "r-tibble",
             "r-tidyr", "r-writexl", "r-gridextra", "r-scales", "r-mlbench",
             "r-mlr3", "r-mlr3learners", "r-ranger", "r-iml", "r-forcats"])
    package_file = TOOLS / "explainer_1.0.3.tar.gz"
    for url in [
        "https://cran.r-project.org/src/contrib/explainer_1.0.3.tar.gz",
        "https://cran.r-project.org/src/contrib/Archive/explainer/explainer_1.0.3.tar.gz",
    ]:
        try:
            urllib.request.urlretrieve(url, package_file)
            break
        except urllib.error.HTTPError as error:
            if error.code != 404:
                raise
    print(execute([MAMBA, "run", "-p", ENV, "R", "CMD", "INSTALL", package_file])[-1500:])
    (TOOLS / "ready").touch()

print(execute([MAMBA, "run", "-p", ENV, R, "--version"]).splitlines()[0])


## 2. Prepare data and train a classifier
Drop the sample ID and omit rows with missing values for this demonstration. The nine original predictors describe cell characteristics, such as clump thickness and uniformity of cell size. The target is the recorded benign or malignant class. Use a stratified split and keep the test observations separate from model training.


In [ ]:
TRAIN_R = r'''
library(explainer)
library(mlr3learners)
RNGkind("L'Ecuyer-CMRG")
seed <- 246L
set.seed(seed)
data("BreastCancer", package = "mlbench")
samples <- na.omit(BreastCancer[, setdiff(names(BreastCancer), "Id")])
task <- mlr3::TaskClassif$new("breast_cytology", samples, target = "Class", positive = "malignant")
task$set_col_roles("Class", add_to = "stratum")
splits <- mlr3::partition(task, ratio = 0.75)
learner <- mlr3::lrn("classif.ranger", predict_type = "prob", num.threads = 2L)
learner$train(task, splits$train)
print(data.frame(complete_samples = nrow(samples), predictors = length(task$feature_names),
           development = length(splits$train), test = length(splits$test)))
'''


## 3. Evaluate held-out predictions
ROC AUC measures discrimination across thresholds. Balanced accuracy averages sensitivity and specificity at the classifier’s decision threshold. Inspect the confusion matrix to see which errors the model made.


In [ ]:
EVALUATE_R = r'''
prediction <- learner$predict(task, splits$test)
scores <- prediction$score(mlr3::msrs(c("classif.auc", "classif.bacc")))
print(round(scores, 3))
write.csv(data.frame(metric = names(scores), value = unname(scores)), "test-metrics.csv", row.names = FALSE)
cm <- explainer::eCM_plot(task = task, trained_model = learner, splits = splits)
ggplot2::ggsave("confusion-matrix.png", cm[[2]], width = 6, height = 5, dpi = 150)
'''


## 4. Explain predictions with SHAP
ExplaineR estimates feature contributions for a sample of held-out observations. Positive contributions increase the model’s predicted probability of the malignant class relative to its baseline. Colour represents the scaled feature value. The marker shape shows whether the prediction was correct. Use the plot to compare model behaviour across samples and examine errors.

The notebook embeds the interactive widget returned by `explainer::eSHAP_plot()`. Hover over a point to inspect it. The same widget is available as a standalone HTML file under Output.


In [ ]:
EXPLAIN_R = r'''
shap <- explainer::eSHAP_plot(task = task, trained_model = learner, splits = splits,
                            sample.size = 30, seed = seed, subset = 0.35)
htmlwidgets::saveWidget(shap[[1]], "interactive-shap.html", selfcontained = TRUE)
'''


## 5. Run the analysis and explore the results
Execute the R sections together to train the model, evaluate the test set, and generate explanations.


In [ ]:
from html import escape
from IPython.display import HTML, Image, display
import shutil

script = WORK / "analysis.R"
script.write_text("\n".join([TRAIN_R, EVALUATE_R, EXPLAIN_R]), encoding="utf-8")
print(execute([MAMBA, "run", "-p", ENV, R, script]))
display(Image(filename=str(WORK / "confusion-matrix.png")))
widget = (WORK / "interactive-shap.html").read_text(encoding="utf-8")
display(HTML('<iframe title="Interactive ExplaineR SHAP plot" sandbox="allow-scripts" '
             'width="100%" height="660" style="border:0" srcdoc="' +
             escape(widget, quote=True) + '"></iframe>'))
shutil.make_archive(str(WORK.parent / "explainer-results"), "zip", root_dir=WORK)
print("Download explainer-results.zip from Output for the figures, metrics, R script, and interactive SHAP plot.")


## 6. Adapt the example
Replace the example data with your own dataset, set the target and positive class, and choose a split that matches your research design. The full ExplaineR tutorial also covers decision curves and subgroup analysis.

**Citation** · Zargari Marandi et al., *ExplaineR*, Bioinformatics Advances. [Methods paper](https://doi.org/10.1093/bioadv/vbae049)
